# SPW on CSCS Clariden: inference, evaluation and comparison with the production run

Runs `scripts/handover_demo.sh` step by step: GraphCast with the production perturbation `graphcast_all`
(sigma 0.01 on all weights), 2023-01-02 00 UTC, 2 members, 24 h, through the production submitter and
container; SwissClim deterministic and probabilistic scores against ERA5; comparison with the production
run of the same init. Every input is local: the initial condition and the truth from the ERA5 zarr in
`/capstor/store/cscs/swissai/weatherbench`, the GraphCast weights from `$STORE/e2s_cache_backup`, the
reference run from `$STORE`. About 15 min.

Needs Unix groups `s83` and `weatherbench`, a Slurm account (`ab016` by default) and `uv` on `PATH`.
Works with any Jupyter kernel (jupyter.cscs.ch, or VS Code on a login node): the work runs in Slurm
jobs and in the repo's own venv. Without Clariden use [spw_quickstart.ipynb](spw_quickstart.ipynb).

In [ ]:
import os
import re
import subprocess
import time
from pathlib import Path

from IPython.display import Image, display

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("AIENS_ACCOUNT", "ab016")  # Slurm account
os.environ.setdefault("PARTITION", "debug")
os.environ["AIENS_REPO"] = str(REPO)
os.environ["AIENS_PY"] = str(REPO / ".venv/bin/python")
SCRATCH = Path(
    os.environ.setdefault("AIENS_SCRATCH", f"/iopsstor/scratch/cscs/{os.environ['USER']}")
)
DEMO_ROOT = Path(os.environ.setdefault("DEMO_ROOT", str(SCRATCH / "aiens_demo")))
# Reference output of the same steps, run on 2026-11-02
REFERENCE = Path("/capstor/store/cscs/mch/s83/sadamov/handover-demo/run")


def run_step(script, step):
    """Run one step of a repo script; wait for the Slurm jobs it submits and report their state."""
    # The kernel's inline matplotlib backend would leak into the jobs (sbatch exports the environment).
    env = {**os.environ, "MPLBACKEND": "Agg"}
    res = subprocess.run(["bash", script, step], cwd=REPO, capture_output=True, text=True, env=env)
    print(res.stdout, res.stderr)
    res.check_returncode()
    for job in re.findall(r"(?:Submitted batch job|->) (\d+)", res.stdout):
        while subprocess.run(
            ["squeue", "-h", "-j", job], capture_output=True, text=True
        ).stdout.strip():
            time.sleep(30)
        state = subprocess.run(
            ["sacct", "-j", job, "-X", "-n", "-o", "State"], capture_output=True, text=True
        ).stdout.split()
        print(f"job {job}: {state[0] if state else 'unknown'}")
        if not state or state[0] != "COMPLETED":
            raise RuntimeError(f"job {job} did not complete, see the log above")

## 1. Environment (once)

Builds `.venv` in the checkout from the pinned lock `env/host-requirements.txt` (about 2 min).

In [ ]:
if not (REPO / ".venv/bin/python").exists():
    run_step("scripts/handover_demo.sh", "venv")

## 2. Inference

One GPU job through `scripts/submit_all_inference.sh` in the GraphCast container (about 4 min).

In [ ]:
run_step("scripts/handover_demo.sh", "infer")

## 3. Evaluation

SwissClim deterministic and probabilistic modules against ERA5 (CPU job, about 7 min).

In [ ]:
run_step("scripts/handover_demo.sh", "eval")

## 4. Comparison with the production run

Same seeds as production, so the members differ only by GPU nondeterminism: expect about 0.04 K RMS in
2m temperature at +24 h, against a member spread of about 0.29 K.

In [ ]:
run_step("scripts/handover_demo.sh", "compare")

In [ ]:
run_step("scripts/handover_demo.sh", "plot")
display(Image(str(DEMO_ROOT / "demo_summary.png")))

## Next steps

- Production grid and evaluation: [docs/HPC.md](../docs/HPC.md) and [docs/REPRODUCING.md](../docs/REPRODUCING.md).
- Where every dataset is: [docs/DATA.md](../docs/DATA.md).